In [7]:
import json
import random
from pathlib import Path

ROOT = Path("../")

def evaluate_policy(file_path, N=10000, max_steps=100, delays=(0, 1)):
    # ===== LOAD =====
    with open(file_path) as f:
        data = json.load(f)

    graph = data["graph"]
    policy = data["policy"]

    adj = graph["adj_matrix"]
    coverage = graph["coverage_matrix"]
    attack_duration = graph["attack_duration"]
    num_nodes = len(adj)

    # ===== HELPERS =====
    def sample_action(dist):
        actions = list(map(int, dist.keys()))
        probs = list(dist.values())
        return random.choices(actions, weights=probs)[0]

    def get_policy(player, def_hist):
        key = f"p={player}|def_hist=" + ",".join(map(str, def_hist))
        return policy.get(key, None)

    # ===== EPISODE =====
    def run_episode(use_trained_attacker=True):
        defender_pos = random.randint(0, num_nodes - 1)
        delay = random.choice(delays)

        def_hist = [defender_pos]

        attacker_target = None
        attack_timer = None

        for t in range(max_steps):

            # attacker
            if t == delay:
                if use_trained_attacker:
                    pol = get_policy(1, def_hist)
                    attacker_target = sample_action(pol) if pol else random.randint(0, num_nodes - 1)
                else:
                    attacker_target = random.randint(0, num_nodes - 1)

                attack_timer = attack_duration[attacker_target]

            # defender
            pol = get_policy(0, def_hist)
            next_pos = sample_action(pol) if pol else random.randint(0, num_nodes - 1)

            if attacker_target is not None:
                cost = adj[defender_pos][next_pos]
                attack_timer -= cost
                if attack_timer < 0:
                    return 0
                
            defender_pos = next_pos
            def_hist.append(defender_pos)

            if attacker_target is not None:
                capture_prob = coverage[defender_pos][attacker_target]

                if random.random() < capture_prob:
                    return 1
                if attack_timer <= 0:
                    return 0
                

        return 0

    # ===== RUN =====
    results = {}

    for mode in [True, False]:
        captures = sum(run_episode(use_trained_attacker=mode) for _ in range(N))
        fails = N - captures

        key = "trained_attacker" if mode else "random_attacker"
        results[key] = {
            "capture": captures / N,
            "fail": fails / N
        }

        print("\n=== MODE:", key.upper(), "===")
        print(f"Rollouts: {N}")
        print(f"Capture: {captures} ({captures/N:.3f})")
        print(f"Fail:    {fails} ({fails/N:.3f})")

    return results

In [64]:
import json
import random
import numpy as np
from pathlib import Path

from open_spiel.python.games import data

ROOT = Path("../")


def evaluate_policy_worst_case_expected(
    file_path,
    rollout_len=1000,
    rollouts_num=100,
    delays=(0, 1),
):

    # ===== LOAD =====

    with open(file_path) as f:
        data = json.load(f)

    graph = data["graph"]
    policy = data["policy"]

    adj = graph["adj_matrix"]
    coverage = graph["coverage_matrix"]
    attack_duration = graph["attack_duration"]
    target_values = graph["targets"]
    attacker_history_length = data["attacker_history_length"]

    num_nodes = len(adj)

    # ===== HELPERS =====

    def sample_action(dist):
        actions = list(map(int, dist.keys()))
        probs = list(dist.values())
        return random.choices(actions, weights=probs)[0]

    def get_policy(player, def_hist):
        key = f"p={player}|def_hist=" + ",".join(map(str, def_hist))
        return policy.get(key, None)

    # ===== GENERATE DEFENDER ROLLOUT =====

    def generate_rollout():

        defender_pos = random.randint(0, num_nodes - 1)

        delay = random.choice(delays)

        rollout = [defender_pos]

        while len(rollout) < rollout_len:

            hist = rollout.copy()

            pol = get_policy(0, hist)

            if pol:
                next_pos = sample_action(pol)
            else:
                next_pos = random.randint(0, num_nodes - 1)

            rollout.append(next_pos)

        return rollout, delay

    # ===== EXPECTED CAPTURE =====

    def expected_capture_probability(path, target):

        survive_prob = 1.0

        for pos in path:
            survive_prob *= (
                1.0 - coverage[pos][target]
            )

        return 1.0 - survive_prob

    # ===== AGGREGATE REWARDS =====

    total_reward = {}
    total_count = {}

    for _ in range(rollouts_num):

        rollout, delay = generate_rollout()
        for target in range(num_nodes):

            if target_values[target] == 0:
                continue

            tau = attack_duration[target]

            full_observation = rollout[:delay + 1]

            if attacker_history_length < 0:
                observation = tuple(full_observation)
            else:
                observation = tuple(
                    full_observation[-attacker_history_length:]
                )

            attack_path = rollout[
                (delay+1):delay + tau + 1
            ]
            # if observation == (9, 3) and target == 5:
            #     print("OBS:", observation)
            #     print("PATH:", attack_path)
            capture_prob = expected_capture_probability(
                attack_path,
                target
            )

            reward = (
                capture_prob *
                target_values[target]
            )

            key = (observation, target)

            total_reward[key] = (
                total_reward.get(key, 0.0)
                + reward
            )

            total_count[key] = (
                total_count.get(key, 0)
                + 1
            )

    # ===== WORST CASE =====

    min_reward = float("inf")
    for key in total_reward:
        if total_count[key] == 0:
            print(key)
    print(total_reward)
    for key in total_reward:

        avg_reward = (
            total_reward[key]
            / total_count[key]
        )

        min_reward = min(
            min_reward,
            avg_reward
        )

    print("\n=== WORST CASE EXPECTED REWARD ===")
    print(
        "Worst-case expected defender reward:",
        round(min_reward, 4)
    )

    return min_reward

In [65]:
file_path = ROOT / "results/full/Gdynia_4.json"

evaluate_policy(str(file_path), N=100000)


=== MODE: TRAINED_ATTACKER ===
Rollouts: 100000
Capture: 56380 (0.564)
Fail:    43620 (0.436)

=== MODE: RANDOM_ATTACKER ===
Rollouts: 100000
Capture: 64049 (0.640)
Fail:    35951 (0.360)


{'trained_attacker': {'capture': 0.5638, 'fail': 0.4362},
 'random_attacker': {'capture': 0.64049, 'fail': 0.35951}}

In [66]:
evaluate_policy_worst_case_expected(
    file_path,
    rollout_len=10,
    rollouts_num=100000
)

{((5,), 2): 3400.5, ((5,), 4): 6379.0, ((5,), 5): 4177.5, ((5,), 6): 4177.5, ((5,), 7): 7578.0, ((5,), 8): 7383.75, ((5,), 9): 6372.875, ((6,), 2): 3369.5, ((6,), 4): 6321.875, ((6,), 5): 4100.375, ((6,), 6): 4100.25, ((6,), 7): 7470.0, ((6,), 8): 7287.75, ((6,), 9): 6316.5, ((1,), 2): 2745.125, ((1,), 4): 4636.5, ((1,), 5): 2265.5, ((1,), 6): 2265.5, ((1,), 7): 4531.0, ((1,), 8): 4622.25, ((1,), 9): 4649.5, ((3,), 2): 11821.75, ((3,), 4): 22755.375, ((3,), 5): 11991.0, ((3,), 6): 12055.75, ((3,), 7): 17660.5, ((3,), 8): 23358.75, ((3,), 9): 22678.625, ((4,), 2): 4073.0, ((4,), 4): 6081.375, ((4,), 5): 4005.75, ((4,), 6): 3991.75, ((4,), 7): 4264.5, ((4,), 8): 5510.5, ((4,), 9): 6754.375, ((7,), 2): 8512.0, ((7,), 4): 14343.5, ((7,), 5): 8350.75, ((7,), 6): 8350.75, ((7,), 7): 13233.0, ((7,), 8): 17489.0, ((7,), 9): 14351.0, ((8,), 2): 2465.0, ((8,), 4): 2487.375, ((8,), 5): 2551.5, ((8,), 6): 2551.75, ((8,), 7): 2557.0, ((8,), 8): 3744.75, ((8,), 9): 2491.625, ((9,), 2): 4040.875, ((9

0.4487331749802059